# FlairSim -- Tutorial Notebook

This notebook demonstrates how to interact with the FlairSim web platform from a Python notebook.
It covers:

1. **Platform status** -- checking the orchestrator is running
2. **Listing scenarios** -- discovering available missions
3. **Creating a session** -- spawning an isolated simulator subprocess
4. **Resetting an episode** -- starting a new mission
5. **Step loop** -- moving the drone with random actions
6. **Visualizing observations** -- decoding and displaying aerial images
7. **Submitting to the leaderboard** -- recording results
8. **Cleanup** -- destroying the session

## Prerequisites

**1. Start the web orchestrator** (in a separate terminal, from the project root):

```bash
cd simulateur

# Using local data in the data/ directory
uv run python -m flairsim.web --scenarios-dir scenarios/ --data-root data/

# Or let scenarios auto-download from HuggingFace (no local data needed)
uv run python -m flairsim.web --scenarios-dir scenarios/
```

The orchestrator runs on `http://127.0.0.1:8080` by default.

**2. Launch this notebook** with `uv` (no separate env needed):

```bash
# JupyterLab (recommended)
uv run --with jupyter --with matplotlib jupyter lab notebooks/tutorial.ipynb

# Or classic notebook
uv run --with jupyter --with matplotlib jupyter notebook notebooks/tutorial.ipynb
```

`uv run --with` adds packages on-the-fly without modifying `pyproject.toml`.
All project dependencies (httpx, Pillow, numpy, etc.) are already available.

## Setup

In [ ]:
import httpx
import base64
import random
import time
import os
from io import BytesIO

# Optional: display images inline
try:
    from PIL import Image as PILImage
    from IPython.display import display, Image, HTML
    HAS_DISPLAY = True
except ImportError:
    HAS_DISPLAY = False
    print("PIL/IPython not available -- images won't be displayed inline")

In [ ]:
# Configuration
ORCHESTRATOR_URL = "http://127.0.0.1:8080"
API_BASE = f"{ORCHESTRATOR_URL}/api"

# API key for AI sessions (load from .env or environment)
API_KEY = os.environ.get("FLAIRSIM_API_KEY", "")

# If not in env, search for .env in current dir and parent dirs
if not API_KEY:
    _dir = os.getcwd()
    for _ in range(5):
        _env = os.path.join(_dir, ".env")
        if os.path.isfile(_env):
            with open(_env) as f:
                for line in f:
                    if line.strip().startswith("FLAIRSIM_API_KEY="):
                        API_KEY = line.strip().split("=", 1)[1].strip()
                        break
            if API_KEY:
                break
        _dir = os.path.dirname(_dir)

# Create HTTP client
client = httpx.Client(base_url=API_BASE, timeout=60.0)

print(f"Orchestrator URL: {ORCHESTRATOR_URL}")
print(f"API key loaded: {'yes' if API_KEY else 'no'} ({len(API_KEY)} chars)")

## 1. Check platform status

In [ ]:
try:
    status = client.get("/status").json()
    print("Platform status:")
    for k, v in status.items():
        print(f"  {k}: {v}")
except httpx.ConnectError:
    print("ERROR: Cannot connect to orchestrator.")
    print(f"Make sure it is running at {ORCHESTRATOR_URL}")
    print("\nStart it with:")
    print("  uv run python -m flairsim.web --scenarios-dir scenarios/ --data-root /path/to/data")

## 2. List available scenarios

Each scenario defines a mission: data source, start position, target location, and prompt templates for VLM agents.

In [ ]:
scenarios_resp = client.get("/scenarios").json()
scenarios = scenarios_resp["scenarios"]

print(f"Found {len(scenarios)} scenario(s):\n")
for s in scenarios:
    stars = '*' * s.get('difficulty', 1)
    print(f"  {s['scenario_id']}")
    print(f"    Name:        {s['name']}")
    print(f"    Description: {s['description'][:80]}...")
    print(f"    Max steps:   {s['max_steps']}")
    print(f"    Difficulty:  {stars}")
    print(f"    Environment: {s.get('environment', [])}")
    print(f"    Modalities:  {s.get('dataset', {}).get('modalities', [])}")
    print()

## 3. Inspect a scenario (with prompt templates)

Scenarios include VLM prompt templates with placeholders for the current state.

In [ ]:
# Pick the first scenario
SCENARIO_ID = scenarios[0]["scenario_id"]
print(f"Selected scenario: {SCENARIO_ID}\n")

scenario_detail = client.get(f"/scenarios/{SCENARIO_ID}").json()

# Display prompt templates (if any)
prompt = scenario_detail.get("prompt", {})
if prompt.get("system"):
    print("=== System Prompt ===")
    print(prompt["system"][:300])
    print()

if prompt.get("user_template"):
    print("=== User Template ===")
    print(prompt["user_template"][:200])
    print()

print(f"Target: ({scenario_detail['target']['x']}, {scenario_detail['target']['y']})")
print(f"Target radius: {scenario_detail['target']['radius']} m")
print(f"Max steps: {scenario_detail['max_steps']}")

## 4. Create a session

This spawns an isolated simulator subprocess on a unique port.

- `mode="human"`: no API key required
- `mode="ai"`: requires `Authorization: Bearer <key>` header

In [ ]:
# Create a session
# Try AI mode if we have a key, fall back to human mode on auth failure
mode = "ai" if API_KEY else "human"

def _create_session(mode, api_key):
    headers = {}
    if mode == "ai" and api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    resp = client.post("/sessions", json={
        "scenario_id": SCENARIO_ID,
        "mode": mode,
        "player_name": "TutorialAgent",
        "model_info": {"type": "random", "version": "1.0"},
    }, headers=headers, timeout=120.0)
    return resp

print(f"Creating {mode} session for {SCENARIO_ID}...")
session_resp = _create_session(mode, API_KEY)

# If AI mode fails (401/503), fall back to human mode
if mode == "ai" and session_resp.status_code in (401, 503):
    print(f"  AI mode failed ({session_resp.status_code}), falling back to human mode...")
    mode = "human"
    session_resp = _create_session(mode, API_KEY)

if session_resp.status_code != 200:
    print(f"ERROR {session_resp.status_code}: {session_resp.text}")
    raise RuntimeError(f"Session creation failed: {session_resp.status_code}")

session = session_resp.json()
SESSION_ID = session["session_id"]

print(f"Session created!")
print(f"  Session ID: {SESSION_ID}")
print(f"  Scenario:   {session['scenario_id']}")
print(f"  Mode:       {session['mode']}")
print(f"  Port:       {session['port']}")
print(f"  Status:     {session['status']}")
print(f"  Base URL:   {session['base_url']}")

## 5. Reset the episode

All simulator interactions go through the proxy: `/api/sessions/{id}/sim/{endpoint}`

In [ ]:
def sim_request(method, endpoint, **kwargs):
    """Helper to send requests to the simulator via the orchestrator proxy."""
    url = f"/sessions/{SESSION_ID}/sim/{endpoint}"
    resp = client.request(method, url, **kwargs)
    resp.raise_for_status()
    return resp.json()


def display_observation(obs, title="Observation"):
    """Display the observation image and key info."""
    ds = obs["drone_state"]
    print(f"\n--- {title} ---")
    print(f"  Step:       {obs['step']}")
    print(f"  Position:   ({ds['x']:.1f}, {ds['y']:.1f})")
    print(f"  Altitude:   {ds['z']:.1f} m")
    print(f"  Footprint:  {obs['ground_footprint']:.1f} m")
    print(f"  GSD:        {obs['ground_resolution']:.3f} m/px")
    print(f"  Distance:   {ds['total_distance']:.1f} m")
    print(f"  Done:       {obs['done']}")
    
    # Display distance to target if available
    meta = obs.get("metadata", {})
    if "distance_to_target" in meta:
        print(f"  To target:  {float(meta['distance_to_target']):.1f} m")
    
    # Display image
    if HAS_DISPLAY and obs.get("image_base64"):
        img_bytes = base64.b64decode(obs["image_base64"])
        img = PILImage.open(BytesIO(img_bytes))
        # Resize for display
        img_resized = img.resize((350, 350))
        display(img_resized)


# Reset the episode
obs = sim_request("POST", "reset")
display_observation(obs, "Initial observation")

## 6. Get simulator configuration

Useful for understanding map bounds, drone limits, etc.

In [ ]:
config = sim_request("GET", "config")

print("Simulator configuration:")
print(f"  Data dir:   {config['data_dir']}")
print(f"  ROI:        {config['roi']}")
print(f"  Tiles:      {config['n_tiles']}")
print(f"  Max steps:  {config['max_steps']}")
print(f"  Running:    {config['is_running']}")
print(f"  Scenario:   {config.get('scenario_id')}")
print()
print("Map bounds:")
mb = config["map_bounds"]
print(f"  X: [{mb['x_min']:.1f}, {mb['x_max']:.1f}] ({mb['width']:.1f} m)")
print(f"  Y: [{mb['y_min']:.1f}, {mb['y_max']:.1f}] ({mb['height']:.1f} m)")
print()
print("Drone limits:")
drone_cfg = config["drone"]
print(f"  Altitude: [{drone_cfg['z_min']}, {drone_cfg['z_max']}] m")
print(f"  Default altitude: {drone_cfg['default_altitude']} m")

## 7. Random agent loop

This demonstrates a simple agent that takes random steps. In a real application,
you would replace the random action generation with a VLM call using the
observation image + prompt template.

In [ ]:
# Reset for a fresh run
obs = sim_request("POST", "reset")

# Store trajectory and step details
trajectory = []
steps_detail = []
start_time = time.time()

# Agent parameters
N_EXPLORE_STEPS = 10    # number of exploration steps before declaring FOUND
STEP_SIZE = 100.0       # large steps (metres) for visible movement

random.seed(42)

print(f"Scenario: {SCENARIO_ID}")
print(f"Plan: {N_EXPLORE_STEPS} random exploration steps, then declare FOUND")
print(f"Step size: {STEP_SIZE} m")
print("=" * 60)

# --- Exploration phase: N random big moves ---
for step_i in range(N_EXPLORE_STEPS):
    ds = obs["drone_state"]
    trajectory.append({"x": ds["x"], "y": ds["y"]})
    
    dx = STEP_SIZE * random.uniform(-1, 1)
    dy = STEP_SIZE * random.uniform(-1, 1)
    
    action = {
        "dx": round(dx, 1),
        "dy": round(dy, 1),
        "dz": 0.0,
        "action_type": "move",
        "reason": f"Random exploration step {step_i + 1}",
    }
    steps_detail.append(action)
    obs = sim_request("POST", "step", json=action)
    
    meta = obs.get("metadata", {})
    dist = float(meta.get("distance_to_target", 0))
    ds = obs["drone_state"]
    print(f"  Step {obs['step']:2d} | "
          f"pos=({ds['x']:.0f}, {ds['y']:.0f}) | "
          f"moved=({action['dx']:+.0f}, {action['dy']:+.0f}) | "
          f"dist_target={dist:.0f}m")
    
    # Show image for first and last exploration step
    if step_i == 0 or step_i == N_EXPLORE_STEPS - 1:
        display_observation(obs, f"Step {obs['step']}")
    
    if obs["done"]:
        break

# --- Declare FOUND to end the episode ---
if not obs["done"]:
    print(f"\nDeclaring FOUND at current position...")
    ds = obs["drone_state"]
    trajectory.append({"x": ds["x"], "y": ds["y"]})
    
    action = {"dx": 0, "dy": 0, "dz": 0, "action_type": "found",
              "reason": "Declaring target found after exploration"}
    steps_detail.append(action)
    obs = sim_request("POST", "step", json=action)

duration = time.time() - start_time

# Final trajectory point
ds = obs["drone_state"]
trajectory.append({"x": ds["x"], "y": ds["y"]})

print("\n" + "=" * 60)
print(f"Episode finished! (done={obs['done']})")

## 8. Analyze results

In [ ]:
result = obs.get("result") or {}

print("Episode Result:")
print(f"  Success:    {result.get('success', 'N/A')}")
print(f"  Reason:     {result.get('reason', 'N/A')}")
print(f"  Steps:      {result.get('steps_taken', 'N/A')}")
dist = result.get('distance_travelled')
print(f"  Distance:   {dist:.1f} m" if dist else "  Distance: N/A")
print(f"  Duration:   {duration:.1f} s")
print(f"  Trajectory: {len(trajectory)} points")

# Display final observation
display_observation(obs, "Final observation")

## 9. Get flight telemetry

The simulator keeps a detailed flight log with every step.

In [ ]:
telemetry = sim_request("GET", "telemetry")

print("Flight telemetry summary:")
print(f"  Total steps:    {telemetry['total_steps']}")
print(f"  Total distance: {telemetry['total_distance']:.1f} m")
print(f"  Clips count:    {telemetry['clips_count']}")
if telemetry.get('altitude_range'):
    print(f"  Altitude range: [{telemetry['altitude_range'][0]:.1f}, "
          f"{telemetry['altitude_range'][1]:.1f}] m")

# Show first 5 records
print(f"\nFirst 5 telemetry records:")
for rec in telemetry["records"][:5]:
    print(f"  step={rec['step']:3d}  pos=({rec['x']:.0f}, {rec['y']:.0f}, {rec['z']:.0f})  "
          f"d=({rec['dx']:.1f}, {rec['dy']:.1f})  clipped={rec['was_clipped']}")

## 10. Plot trajectory (optional)

If `matplotlib` is available, visualize the drone's path.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    
    xs = [p["x"] for p in trajectory]
    ys = [p["y"] for p in trajectory]
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    
    # Plot trajectory
    ax.plot(xs, ys, 'c-', linewidth=0.8, alpha=0.7, label='Trajectory')
    ax.plot(xs[0], ys[0], 'go', markersize=12, label='Start', zorder=5)
    ax.plot(xs[-1], ys[-1], 'rs', markersize=12, label='End', zorder=5)
    
    # Plot target
    target = scenario_detail["target"]
    circle = patches.Circle(
        (target["x"], target["y"]), target["radius"],
        linewidth=2, edgecolor='lime', facecolor='lime',
        alpha=0.2, label=f'Target (r={target["radius"]}m)'
    )
    ax.add_patch(circle)
    ax.plot(target["x"], target["y"], 'g+', markersize=15, 
            markeredgewidth=2, zorder=5)
    
    # Map bounds
    mb = config["map_bounds"]
    ax.set_xlim(mb["x_min"], mb["x_max"])
    ax.set_ylim(mb["y_min"], mb["y_max"])
    ax.set_aspect('equal')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    ax.set_title(f'Drone trajectory -- {SCENARIO_ID}')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

except ImportError:
    print("matplotlib not available -- skipping trajectory plot")
    print("Install with: pip install matplotlib")

## 11. Submit result to leaderboard

In [ ]:
run_data = {
    "scenario_id": SCENARIO_ID,
    "mode": mode,
    "session_id": SESSION_ID,
    "player_name": "TutorialAgent",
    "model_name": "Random Walker v1.0",
    "success": result.get("success", False),
    "reason": result.get("reason", ""),
    "steps_taken": result.get("steps_taken"),
    "distance_travelled": result.get("distance_travelled"),
    "duration_s": round(duration, 2),
    "trajectory": trajectory,
    "steps_detail": steps_detail,
    "model_info": {"type": "random", "step_size": STEP_SIZE, "seed": 42},
}

lb_resp = client.post("/leaderboard/submit", json=run_data)
lb_result = lb_resp.json()

print(f"Submitted to leaderboard!")
print(f"  Run ID: {lb_result['run_id']}")

## 12. View the leaderboard

In [ ]:
leaderboard = client.get("/leaderboard", params={"scenario_id": SCENARIO_ID}).json()

print(f"Leaderboard for {SCENARIO_ID}:")
print(f"{'Rank':>4}  {'Player':<20}  {'Success':>7}  {'Steps':>5}  {'Distance':>8}  {'Mode':<5}")
print("-" * 65)

for i, run in enumerate(leaderboard["runs"][:10], 1):
    name = run.get("player_name") or run.get("model_name") or "anonymous"
    success = "YES" if run["success"] else "no"
    steps = run.get("steps_taken", "?")
    dist = f"{run['distance_travelled']:.0f}m" if run.get('distance_travelled') else "?"
    print(f"{i:4d}  {name:<20}  {success:>7}  {steps:>5}  {dist:>8}  {run['mode']:<5}")

## 13. Cleanup -- destroy the session

This kills the simulator subprocess and frees the port.

In [ ]:
destroy_resp = client.delete(f"/sessions/{SESSION_ID}")
print(f"Session destroyed: {destroy_resp.json()}")

## 14. List active sessions

Verify that our session has been cleaned up.

In [ ]:
sessions = client.get("/sessions").json()
print(f"Active sessions: {len(sessions['sessions'])}")
for s in sessions["sessions"]:
    print(f"  {s['session_id'][:8]}... | {s['scenario_id']} | {s['mode']} | {s['status']}")

---

## Going further

### Using a VLM agent

Replace the random action generation with a VLM call:

```python
# Get the prompt template from the scenario
system_prompt = scenario_detail["prompt"]["system"]
user_template = scenario_detail["prompt"]["user_template"]

# At each step, fill the template with current state
user_prompt = user_template.format(
    x=obs["drone_state"]["x"],
    y=obs["drone_state"]["y"],
    z=obs["drone_state"]["z"],
    step=obs["step"],
    max_steps=config["max_steps"],
    steps_remaining=config["max_steps"] - obs["step"],
    distance=obs["drone_state"]["total_distance"],
)

# Send system_prompt + user_prompt + obs["image_base64"] to your VLM
# Parse the VLM response into {"dx": ..., "dy": ..., "action_type": ...}
```

### Multi-modality images

```python
# Access per-modality images
for modality, b64 in obs["images"].items():
    img_bytes = base64.b64decode(b64)
    print(f"{modality}: {len(img_bytes)} bytes")
```

### Grid overlay for VLM spatial prompting

```python
# Request images with a 4x4 grid overlay
obs = sim_request("POST", "reset?grid=4")
obs = sim_request("POST", "step?grid=4", json={"dx": 10, "dy": 0})
# Image will have A1..D4 cell labels overlaid
```

### Watching a session from the browser

While this notebook runs, open the web platform and use the **Spectator** mode
to watch the drone move in real time.

### Direct simulator API (no orchestrator)

```python
# Connect directly to a standalone simulator
direct = httpx.Client(base_url="http://127.0.0.1:8000")
obs = direct.post("/reset").json()
obs = direct.post("/step", json={"dx": 10, "dy": 0}).json()
```

In [ ]:
# Close the HTTP client
client.close()
print("Done! Client closed.")